# 00 · Carga y verificación de datos

Este notebook prepara un punto de entrada reproducible para el proyecto **CommonLit - Evaluate Student Summaries**. Localiza la raíz del repositorio, verifica los archivos necesarios para el EDA, carga los CSV sin modificar los datos originales y documenta su estructura.

**Alcance:** para el análisis exploratorio sólo son obligatorios `summaries_train.csv` y `prompts_train.csv`. Los archivos de prueba y `sample_submission.csv` son opcionales y se reportan sin bloquear la ejecución.

## 1. Entorno y rutas

In [1]:
from pathlib import Path
import platform
import sys

import matplotlib
import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. Ejecute el notebook dentro del repositorio."
    )


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    OPTIONAL_COMPETITION_FILES,
    REQUIRED_TRAIN_FILES,
    load_csv_files,
    missing_files,
)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

for directory in (RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Datos originales:   {RAW_DATA_DIR}")

Raíz del proyecto: C:\Users\richi\Documents\2026_S2_Local\Data_Science\proyecto2_DS
Datos originales:   C:\Users\richi\Documents\2026_S2_Local\Data_Science\proyecto2_DS\data\raw


In [2]:
versions = pd.Series(
    {
        "Python": platform.python_version(),
        "pandas": pd.__version__,
        "NumPy": np.__version__,
        "Matplotlib": matplotlib.__version__,
    },
    name="versión",
)
display(versions.to_frame())

,versión
Python,3.12.10
pandas,2.3.3
NumPy,2.2.6
Matplotlib,3.10.9


## 2. Inventario de archivos

Se separan los archivos imprescindibles para el EDA de los archivos opcionales de la competencia. Esta distinción permite reproducir el análisis aun cuando sólo se hayan descargado los datos de entrenamiento.

In [3]:
file_roles = {
    "summaries_train.csv": ("Resúmenes con variables objetivo", True),
    "prompts_train.csv": ("Prompts asociados a entrenamiento", True),
    "summaries_test.csv": ("Resúmenes de prueba de Kaggle", False),
    "prompts_test.csv": ("Prompts de prueba de Kaggle", False),
    "sample_submission.csv": ("Formato de envío a Kaggle", False),
}

inventory_rows = []
for filename, (role, required) in file_roles.items():
    path = RAW_DATA_DIR / filename
    inventory_rows.append(
        {
            "archivo": filename,
            "función": role,
            "obligatorio_para_EDA": required,
            "disponible": path.is_file(),
            "tamaño_MiB": round(path.stat().st_size / 1024**2, 3) if path.is_file() else np.nan,
        }
    )

inventory = pd.DataFrame(inventory_rows)
display(inventory)
inventory.to_csv(TABLES_DIR / "00_data_inventory.csv", index=False)

,archivo,función,obligatorio_para_EDA,disponible,tamaño_MiB
0,summaries_train.csv,Resúmenes con variables objetivo,True,True,3.275
1,prompts_train.csv,Prompts asociados a entrenamiento,True,True,0.015
2,summaries_test.csv,Resúmenes de prueba de Kaggle,False,True,0.000
3,prompts_test.csv,Prompts de prueba de Kaggle,False,True,0.000
4,sample_submission.csv,Formato de envío a Kaggle,False,True,0.000


In [4]:
missing_required = missing_files(RAW_DATA_DIR, REQUIRED_TRAIN_FILES)
missing_optional = missing_files(RAW_DATA_DIR, OPTIONAL_COMPETITION_FILES)

if missing_required:
    raise FileNotFoundError(
        "Faltan archivos indispensables para el EDA: "
        + ", ".join(missing_required)
        + ". Consulte data/raw/README.md para descargarlos."
    )

print("✓ Los dos archivos de entrenamiento requeridos están disponibles.")
if missing_optional:
    print(
        "ℹ Archivos opcionales no disponibles (no bloquean el EDA): "
        + ", ".join(missing_optional)
    )

✓ Los dos archivos de entrenamiento requeridos están disponibles.


### Descarga opcional

Si faltan los archivos requeridos, la descarga debe hacerse de forma explícita y con las credenciales de Kaggle configuradas. La celda siguiente está desactivada por defecto para evitar accesos de red inesperados.

In [5]:
DOWNLOAD_FROM_KAGGLE = False

if DOWNLOAD_FROM_KAGGLE:
    import kagglehub

    download_path = kagglehub.competition_download(
        "commonlit-evaluate-student-summaries",
        output_dir=str(RAW_DATA_DIR),
    )
    print(f"Descarga finalizada en: {download_path}")
else:
    print("Descarga omitida. Cambie DOWNLOAD_FROM_KAGGLE a True sólo si es necesaria.")

Descarga omitida. Cambie DOWNLOAD_FROM_KAGGLE a True sólo si es necesaria.


## 3. Carga de los datos disponibles

In [6]:
available_files = [
    filename
    for filename in (*REQUIRED_TRAIN_FILES, *OPTIONAL_COMPETITION_FILES)
    if (RAW_DATA_DIR / filename).is_file()
]

datasets = load_csv_files(RAW_DATA_DIR, available_files)
summaries_train = datasets["summaries_train"]
prompts_train = datasets["prompts_train"]

print("Datasets cargados: " + ", ".join(datasets))

Datasets cargados: summaries_train, prompts_train, summaries_test, prompts_test, sample_submission


In [7]:
dimensions = pd.DataFrame(
    [
        {
            "dataset": name,
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "memoria_MiB": round(df.memory_usage(deep=True).sum() / 1024**2, 3),
        }
        for name, df in datasets.items()
    ]
).sort_values("dataset", ignore_index=True)

display(dimensions)
dimensions.to_csv(TABLES_DIR / "00_dataset_dimensions.csv", index=False)

,dataset,filas,columnas,memoria_MiB
0,prompts_test,2,4,0.001
1,prompts_train,4,4,0.031
2,sample_submission,4,3,0.000
3,summaries_test,4,3,0.001
4,summaries_train,7165,5,4.449


## 4. Contrato de esquema

La validación temprana del nombre y orden de las columnas evita que los notebooks posteriores produzcan resultados silenciosamente incorrectos.

In [8]:
EXPECTED_COLUMNS = {
    "summaries_train": ["student_id", "prompt_id", "text", "content", "wording"],
    "prompts_train": ["prompt_id", "prompt_question", "prompt_title", "prompt_text"],
}

for name, expected_columns in EXPECTED_COLUMNS.items():
    actual_columns = datasets[name].columns.tolist()
    assert actual_columns == expected_columns, (
        f"Esquema inesperado en {name}. "
        f"Esperado: {expected_columns}; recibido: {actual_columns}"
    )

print("✓ Los esquemas de entrenamiento coinciden con el contrato esperado.")

✓ Los esquemas de entrenamiento coinciden con el contrato esperado.


In [9]:
schema = pd.concat(
    [
        pd.DataFrame(
            {
                "dataset": name,
                "variable": df.columns,
                "tipo": df.dtypes.astype(str).values,
                "no_nulos": df.notna().sum().values,
                "valores_únicos": df.nunique(dropna=False).values,
            }
        )
        for name, df in datasets.items()
    ],
    ignore_index=True,
)

display(schema)
schema.to_csv(TABLES_DIR / "00_dataset_schema.csv", index=False)

,dataset,variable,tipo,no_nulos,valores_únicos
0,summaries_train,student_id,object,7165,7165
1,summaries_train,prompt_id,object,7165,4
2,summaries_train,text,object,7165,7165
3,summaries_train,content,float64,7165,1134
4,summaries_train,wording,float64,7165,1134
5,prompts_train,prompt_id,object,4,4
6,prompts_train,prompt_question,object,4,4
7,prompts_train,prompt_title,object,4,4
8,prompts_train,prompt_text,object,4,4
9,summaries_test,student_id,object,4,4


## 5. Vista previa

Se muestran sólo registros de entrenamiento. Los datos originales permanecen inmutables en `data/raw/`.

In [10]:
display(summaries_train.head(3))

,student_id,prompt_id,text,content,wording
0,000e8c3c7ddb,814d6b,The third wave was an experimentto see how peo...,0.205683,0.380538
1,0020ae56ffbf,ebad26,They would rub it up with soda to make the sme...,-0.548304,0.506755
2,004e978e639e,3b9047,"In Egypt, there were many occupations and soci...",3.128928,4.231226


In [11]:
display(prompts_train[["prompt_id", "prompt_title", "prompt_question"]])

,prompt_id,prompt_title,prompt_question
0,39c16e,On Tragedy,Summarize at least 3 elements of an ideal trag...
1,3b9047,Egyptian Social Structure,"In complete sentences, summarize the structure..."
2,814d6b,The Third Wave,Summarize how the Third Wave developed over su...
3,ebad26,Excerpt from The Jungle,Summarize the various ways the factory would u...


## Conclusión

La carga queda preparada para los notebooks siguientes: los archivos de entrenamiento se validan como obligatorios, los archivos de competencia se tratan como opcionales, los esquemas se verifican mediante aserciones y el inventario reproducible se guarda en `outputs/tables/`. El notebook `01_data_quality.ipynb` continuará con nulos, duplicados, integridad referencial, texto y variables objetivo.